In [1]:
!pip install sentence-transformers faiss-cpu transformers PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 38.3 MB/s eta 0:00:00


In [2]:
import os
import fitz  # PyMuPDF
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from google.colab import files


In [3]:
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded file:", pdf_path)


Saving 01. Neuro-Linguistic Programming, The Key To Accelerated Learning Author Peter EH Smee and Linda Smee.pdf to 01. Neuro-Linguistic Programming, The Key To Accelerated Learning Author Peter EH Smee and Linda Smee.pdf
Uploaded file: 01. Neuro-Linguistic Programming, The Key To Accelerated Learning Author Peter EH Smee and Linda Smee.pdf


In [4]:
def extract_text_from_pdf(file_path):
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

raw_text = extract_text_from_pdf(pdf_path)
print(raw_text[:500])  # Preview first 500 chars


Authored by:  
Dr Peter EH Smee  
with Linda Smee   
 
 
 
 
Neuro-Linguistic Programming,  
The Key To Accelerated Learning 
A guide for students, who want to learn quickly; and for teachers, 
trainers and coaches who desire to teach their lessons with 
more fun, less effort and better results 
 
 
 
Copyright Material 
Dr Peter EH Smee and Linda Smee 
circle-of-excellence.com, November 2002 
 
 
  
 
Introduction, Page 2
A note about style 
 
The style of writing in this book is deliberately i


In [5]:
def clean_and_split(text, chunk_size=500):
    text = text.replace('\n', ' ').replace('\xa0', ' ')
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks

documents = clean_and_split(raw_text)
print(f"Total chunks: {len(documents)}")


Total chunks: 239


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(documents)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
dimension = doc_embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))
print("FAISS index created and populated.")


FAISS index created and populated.


In [8]:
def retrieve_chunks(query, top_k=3):
    query_vec = model.encode([query])
    distances, indices = index.search(np.array(query_vec), top_k)
    results = [documents[i] for i in indices[0]]
    return results


In [9]:
qa_pipeline = pipeline("text-generation", model="distilgpt2")

user_query = input("Ask your question: ")
relevant_chunks = retrieve_chunks(user_query)

context = " ".join(relevant_chunks)
prompt = f"Context: {context}\nQuestion: {user_query}\nAnswer:"

response = qa_pipeline(prompt, max_length=100, do_sample=True)[0]['generated_text']
print("\n💬 Final Answer:\n", response)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


Ask your question: How does camera calibration improve inspection accuracy?


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💬 Final Answer:
 Context:  realise that, for you, you feel that you have learned something when  you can generate a large mental image, as if that image were projected onto a  screen 2 metres to your right. You are also aware that the image has very  specific size and colour tones. You can then use that information in a very  practical way. Now that you have ‘calibrated’ your own sense of what it is to  understand a new idea, whenever you begin to study something new you will  immediately be drawn to methods of study tha lassroom. Calibration of current abilities, generating a benchmark from which to calibrate future  success. Applying the VAK Model, to the skills involved in reading and realise what your current  reading strategies are. Applying the VAK Model, to enhance fast 'visual processing' of texts. Re- engaging with the conscious, verbal' mind, using techniques that enable you to derive  "conscious verbal" understanding from the "visual subconscious" mental processing of a wri

In [10]:
print("✅ Pipeline Summary")
print("""
1. User uploaded a PDF file.
2. Text was extracted and split into chunks.
3. Embeddings were generated using SentenceTransformer.
4. Embeddings stored in FAISS for fast vector similarity search.
5. On query, top chunks were retrieved.
6. HuggingFace LLM used to generate a contextual answer.

Tools Used:
- SentenceTransformer (MiniLM)
- FAISS (vector store)
- DistilGPT2 (language model)
- PyMuPDF (PDF parsing)
- Google Colab (environment)
""")


✅ Pipeline Summary

1. User uploaded a PDF file.
2. Text was extracted and split into chunks.
3. Embeddings were generated using SentenceTransformer.
4. Embeddings stored in FAISS for fast vector similarity search.
5. On query, top chunks were retrieved.
6. HuggingFace LLM used to generate a contextual answer.

Tools Used:
- SentenceTransformer (MiniLM)
- FAISS (vector store)
- DistilGPT2 (language model)
- PyMuPDF (PDF parsing)
- Google Colab (environment)

